### Modeling with XGBoost


In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error

In [2]:
# ---- Load data ----
train = pd.read_csv('../data preprocessing+EDA/train_updated.csv')
test = pd.read_csv('../data preprocessing+EDA/test_updated.csv')

In [3]:
FEATURES = [
    'Urban_rura_Urban', 
    'LU_pct_micro_2025_msoa', 
    'LU_pct_large_2025_msoa',
    'log_enterprises_per_1k_residents_2025', 
    'turnover_diversity_1-HHI_2025_msoa',
    'LU_diversity_1-HHI_2025', 
    'share_enterprises_kibs_2025_msoa',
    'emp_rate', 
    'full_time_share', 
    'log_Mid-2024 population',
    'IMD_decile'
]

TARGETS = ['log_total_GVA_2023', 'log_gva_per_worker_2023', 'log_GVA_2025_predicted']

PARAM_DIST = {
    'n_estimators': [100, 200, 300, 500, 800],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.05, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.5, 1, 1.5, 2],
}

In [4]:
RANDOM_STATE = 42

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [5]:
results = {}
fitted_models = {}
 
for target in TARGETS:
    X_train, y_train = train[FEATURES], train[target]
    X_test, y_test = test[FEATURES], test[target]
 
    base_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
 
    search = RandomizedSearchCV(
        base_model,
        param_distributions=PARAM_DIST,
        n_iter=50,
        scoring='r2',
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
 
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
 
    results[target] = {
        'best_params': search.best_params_,
        'cv_best_r2': search.best_score_,
        'test_r2': r2_score(y_test, y_pred),
        'test_rmse': root_mean_squared_error(y_test, y_pred),
        'test_mae': mean_absolute_error(y_test, y_pred),
        'test_mape': mean_absolute_percentage_error(y_test, y_pred),
    }
    fitted_models[target] = best_model
 
    print(f"\n=== {target} ===")
    print(f"Best params: {search.best_params_}")
    print(f"CV best R2:  {search.best_score_:.4f}")
    print(f"Test R2:     {results[target]['test_r2']:.4f}")
    print(f"Test RMSE:   {results[target]['test_rmse']:.4f}")
    print(f"Test MAE:    {results[target]['test_mae']:.4f}")
    print(f"Test MAPE:   {results[target]['test_mape']:.4f}")
 
results_df = pd.DataFrame(results).T
print("\n=== Summary across all three targets ===")
print(results_df[['cv_best_r2', 'test_r2', 'test_rmse', 'test_mae', 'test_mape']])


=== log_total_GVA_2023 ===
Best params: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.9}
CV best R2:  0.7282
Test R2:     0.5878
Test RMSE:   0.7080
Test MAE:    0.4294
Test MAPE:   0.1183

=== log_gva_per_worker_2023 ===
Best params: {'subsample': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
CV best R2:  0.2932
Test R2:     0.2089
Test RMSE:   0.6865
Test MAE:    0.4090
Test MAPE:   0.0365

=== log_GVA_2025_predicted ===
Best params: {'subsample': 0.7, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.9}
CV best R2:  0.7232
Test R2:     0.5986
Test RMSE:   0.7147
Test MAE:    0.4419
Test MAPE:   0.1227

=== Summary across all three targets ===
                        cv_best_r2   tes